In [8]:
import os
import pandas as pd
import numpy as np
import re
from pathlib import Path
import shutil
from datetime import datetime

In [2]:
current_path = os.getcwd()
print("Current:", current_path)
workspace = current_path
os.chdir(workspace)
print("Work:", os.getcwd())

Current: /home/ssm-user/project/scores
Work: /home/ssm-user/project/scores


In [32]:
df1 = pd.read_csv("batch1_ligand_and_affinity_cnn.txt", sep="\t")
df2 = pd.read_csv("batch_2_ligand_and_affinity_and_cnn.txt", sep="\t")

df33 = pd.read_csv("batch_3_ligand_and_affinity_autodock.txt", sep="\t")
df3 = pd.read_csv("batch_3_ligand_and_affinity_rescoring_gnina.txt", sep="\t")

df4 = pd.read_csv("batch_4_ligand_and_affinity_autodock_rescoring_final.txt", sep="\t")
df5 = pd.read_csv("batch_5_ligand_and_affinity_and_cnn.txt",
    delim_whitespace=True,
    engine="python",
    on_bad_lines="skip")

/tmp/ipykernel_998/2073837860.py:8: FutureWarning: The 'delim_whitespace' keyword in pd.read_csv is deprecated and will be removed in a future version. Use ``sep='\s+'`` instead
  df5 = pd.read_csv(


In [33]:
print("df1 columns:", df1.columns.tolist())
print("df2 columns:", df2.columns.tolist())
print("df33 columns:", df33.columns.tolist())
print("df3 columns:", df3.columns.tolist())
print("df4 columns:", df4.columns.tolist())
print("df5 columns:", df5.columns.tolist())

df1 columns: ['Rank', 'Ligand', 'SMILES', 'VinaAffinity', 'CNNscore', 'CNNaffinity']
df2 columns: ['Rank', 'Ligand', 'SMILES', 'Binding Affinity (kcal/mol)', 'CNNscore', 'CNNaffinity']
df33 columns: ['Rank', 'Ligand', 'SMILES', 'Binding_Affinity(kcal/mol)']
df3 columns: ['Rank', 'Ligand', 'SMILES', 'CNN_affinity', 'CNN_score']
df4 columns: ['Rank', 'LigandID', 'SMILES', 'AutoDockAffinity(kcal/mol)', 'GNINAaffinity(kcal/mol)', 'CNNscore']
df5 columns: ['Rank', 'Ligand', 'SMILES', 'Gnina_affinity(kcal/mol)', 'CNN_pose_score', 'CNN_affinity']


In [25]:
df1_re = df1[['Ligand', 'SMILES', 'VinaAffinity', 'CNNaffinity']]
df1_re = df1_re.dropna()

print(df1.shape)
print(df1_re.shape)

df1_re = df1_re.rename(columns={
    'Ligand': 'ligand',
    'SMILES': 'smiles',
    'VinaAffinity': 'vina_affinity',
    'CNNaffinity': 'gnina_affinity'
})

(20032, 6)
(20032, 4)


In [26]:
df2_re = df2[['Ligand', 'SMILES', 'Binding Affinity (kcal/mol)', 'CNNaffinity']]
df2_re = df2_re.dropna()

print(df2.shape)
print(df2_re.shape)

df2_re = df2_re.rename(columns={
    'Ligand': 'ligand',
    'SMILES': 'smiles',
    'Binding Affinity (kcal/mol)': 'vina_affinity',
    'CNNaffinity': 'gnina_affinity'
})

(18893, 6)
(18893, 4)


In [27]:
df4_re = df4[['LigandID', 'SMILES', 'AutoDockAffinity(kcal/mol)', 'GNINAaffinity(kcal/mol)']]
df4_re = df4_re.dropna()

print(df4.shape)
print(df4_re.shape)

df4_re = df4_re.rename(columns={
    'LigandID': 'ligand',
    'SMILES': 'smiles',
    'AutoDockAffinity(kcal/mol)': 'vina_affinity',
    'GNINAaffinity(kcal/mol)': 'gnina_affinity'
})

(20032, 6)
(19561, 4)


In [30]:
df3_re1 = df33[['Ligand', 'SMILES', 'Binding_Affinity(kcal/mol)']]
df3_re2 = df3[['Ligand', 'SMILES', 'CNN_affinity']]

df3_re = pd.merge(
    df3_re1,
    df3_re2,
    on=['Ligand', 'SMILES'],
    how='inner'
)

df3_re = df3_re.dropna()

print(df3.shape)
print(df3_re.shape)

df3_re = df3_re.rename(columns={
    'LigandID': 'ligand',
    'SMILES': 'smiles',
    'Binding_Affinity(kcal/mol)': 'vina_affinity',
    'CNN_affinity': 'gnina_affinity'
})

(20023, 5)
(20023, 4)


In [35]:
df5_re = df5[['Ligand', 'SMILES', 'Gnina_affinity(kcal/mol)', 'CNN_affinity']]
df5_re = df5_re.dropna()

print(df5.shape)
print(df5_re.shape)

df5_re = df5_re.rename(columns={
    'Ligand': 'ligand',
    'SMILES': 'smiles',
    'Gnina_affinity(kcal/mol)': 'vina_affinity',
    'CNN_affinity': 'gnina_affinity'
})

(19468, 6)
(19447, 4)


In [36]:
df5_re

,ligand,smiles,vina_affinity,gnina_affinity
0,Z29672623,CC1=NN=C(NC(=O)C=2C=CC=C(C2)S(=O)(=O)N(CC=3C=C...,-11.00,8.319
1,Z414953952,CCN(C(=O)C=1C=C(C)C(OCC=2C=CC=NC2)=C(C)C1)C=3C...,-8.24,8.268
2,Z115170498,CN1C=CC(=N1)NC(=O)C=2C=CC=C(C2)S(=O)(=O)N(CC=3...,-11.07,8.242
3,Z2627253727,O=C(CC12CCCCC1C2(F)F)NC=3C=CC=CC3N4N=CC=5C(=O)...,-10.46,8.239
4,Z28500484,CCCN(CC1=NC=2C=C(Cl)C=CC2C(=O)N1)C(=O)C=3C=CC4...,-9.92,8.225
...,...,...,...,...
19442,Z3346939991,N#CC=1C=CC(=CC1)N2CCCC(O)(C2)C(=O)O,-7.45,5.110
19443,Z749331484,O=C(NCC1COCCO1)N2CCN(CC(F)(F)F)CC2,-7.43,5.110
19444,Z2057690529,O=C(CN1C[C@@H](F)C[C@H]1CO)N2CCNC(=O)C2,-6.82,5.045
19445,Z1444767475,COCC1=CC=C(O1)C(=O)N2CCOCC2C(=O)O,-6.64,5.001


In [37]:
df1_re.to_csv("batch_1_score.txt",sep="\t", index=False)
df2_re.to_csv("batch_2_score.txt",sep="\t", index=False)
df3_re.to_csv("batch_3_score.txt",sep="\t", index=False)
df4_re.to_csv("batch_4_score.txt",sep="\t", index=False)
df5_re.to_csv("batch_5_score.txt",sep="\t", index=False)